# 06 — Bases de données et SQL (`sqlite3`)

Modèle relationnel, requêtes et agrégations. *(Notion complémentaire, absente des sources initiales.)*

Chaque notion suit le schéma : **définition → quand l'utiliser → code testé (`assert`) → pièges → astuce**.

> Toutes les requêtes s'exécutent sur une base **en mémoire** (`":memory:"`), détruite à la fin.


## 6.1 Connexion et création des tables

**Définition.** Une base relationnelle stocke des lignes dans des tables ; `sqlite3.connect(":memory:")` crée une base volatile.

**Quand l'utiliser.** Pour s'entraîner ou tester sans écrire sur le disque.


In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE eleves (
    id     INTEGER PRIMARY KEY,
    nom    TEXT    NOT NULL,
    classe TEXT    NOT NULL,
    age    INTEGER NOT NULL
);

CREATE TABLE notes (
    id       INTEGER PRIMARY KEY,
    eleve_id INTEGER NOT NULL,
    matiere  TEXT    NOT NULL,
    note     REAL    NOT NULL,
    FOREIGN KEY (eleve_id) REFERENCES eleves(id)
);
""")
conn.commit()

tables = {ligne[0] for ligne in cur.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table'"
)}
assert tables == {"eleves", "notes"}
print("tables :", sorted(tables))


**Pièges.** Oublier `conn.commit()` après une écriture ; `:memory:` disparaît à la fermeture de la connexion.

**Astuce.** `PRIMARY KEY` garantit l'unicité ; `NOT NULL` empêche les valeurs manquantes.


## 6.2 Insérer des données (`INSERT`)

**Définition.** `INSERT INTO table (...) VALUES (...)` ajoute une ligne ; `executemany` en ajoute plusieurs.

**Quand l'utiliser.** Alimenter une table ; toujours utiliser des paramètres `?` plutôt que la concaténation.


In [ ]:
eleves = [
    (1, "Amine",   "MPSI", 18),
    (2, "Basma",   "MPSI", 17),
    (3, "Chaimae", "PCSI", 18),
    (4, "Driss",   "PCSI", 19),
    (5, "Elsa",    "PSI",  17),
    (6, "Farah",   "PSI",  18),
]
cur.executemany("INSERT INTO eleves (id, nom, classe, age) VALUES (?, ?, ?, ?)", eleves)

notes = [
    (1, 1, "Maths", 16.5),
    (2, 1, "Info",  18.0),
    (3, 2, "Maths", 14.0),
    (4, 2, "Info",  15.5),
    (5, 3, "Maths", 19.0),
    (6, 4, "Maths", 12.0),
    (7, 4, "Info",  17.0),
    (8, 5, "Info",  20.0),
    (9, 5, "Maths", 11.0),
]
cur.executemany("INSERT INTO notes (id, eleve_id, matiere, note) VALUES (?, ?, ?, ?)", notes)
conn.commit()

assert cur.execute("SELECT COUNT(*) FROM eleves").fetchone()[0] == 6
assert cur.execute("SELECT COUNT(*) FROM notes").fetchone()[0] == 9


**Pièges.** Concaténer les valeurs dans la requête (injection SQL) ; insérer un `eleve_id` inexistant (SQLite ne vérifie pas les clés étrangères par défaut).

**Astuce.** `executemany` est plus rapide qu'une boucle de `execute` + `commit` à chaque ligne.


## 6.3 `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`

**Définition.** `SELECT` choisit des colonnes, `WHERE` filtre les lignes, `ORDER BY` trie, `LIMIT` coupe.

**Quand l'utiliser.** Toute extraction de données ; `WHERE` évite de tout charger en mémoire.


In [ ]:
mpsi = cur.execute(
    "SELECT nom FROM eleves WHERE classe = ? ORDER BY nom", ("MPSI",)
).fetchall()
assert mpsi == [("Amine",), ("Basma",)]

# tri décroissant + limite
top3 = cur.execute(
    "SELECT nom, age FROM eleves ORDER BY age DESC, nom ASC LIMIT 3"
).fetchall()
assert top3 == [("Driss", 19), ("Amine", 18), ("Chaimae", 18)]

# valeurs distinctes
classes = cur.execute("SELECT DISTINCT classe FROM eleves ORDER BY classe").fetchall()
assert classes == [("MPSI",), ("PCSI",), ("PSI",)]

# motif LIKE
a_commence = cur.execute("SELECT nom FROM eleves WHERE nom LIKE 'A%'").fetchall()
assert a_commence == [("Amine",)]


**Pièges.** `WHERE age = NULL` ne renvoie jamais rien : il faut `IS NULL`. `ORDER BY` sans critère stable peut varier entre exécutions.

**Astuce.** `ORDER BY age DESC, nom ASC` départage les égalités de façon déterministe.


## 6.4 Agrégats : `COUNT`, `AVG`, `MIN`, `MAX`, `SUM`

**Définition.** Les fonctions d'agrégation réduisent un ensemble de lignes à une valeur.

**Quand l'utiliser.** Statistiques globales (moyenne, total, extrêmes).


In [ ]:
stats = cur.execute(
    "SELECT COUNT(*), AVG(note), MIN(note), MAX(note), SUM(note) FROM notes"
).fetchone()

assert stats[0] == 9
assert abs(stats[1] - 143 / 9) < 1e-9     # moyenne = 15,888…
assert stats[2] == 11.0 and stats[3] == 20.0
assert abs(stats[4] - 143.0) < 1e-9


**Pièges.** `COUNT(*)` compte les lignes, `COUNT(colonne)` ignore les `NULL` ; `AVG` renvoie `NULL` sur un ensemble vide.

**Astuce.** `AVG` ne renvoie pas un entier : comparer avec une tolérance (`abs(...) < 1e-9`).


## 6.5 `GROUP BY` et `HAVING`

**Définition.** `GROUP BY` regroupe les lignes par valeur ; `HAVING` filtre les **groupes** (alors que `WHERE` filtre les lignes).

**Quand l'utiliser.** Moyenne par matière, nombre d'élèves par classe, groupes au-dessus d'un seuil.


In [ ]:
par_matiere = cur.execute(
    "SELECT matiere, AVG(note) FROM notes GROUP BY matiere ORDER BY matiere"
).fetchall()
assert par_matiere[0][0] == "Info"
assert abs(par_matiere[0][1] - 17.625) < 1e-9
assert par_matiere[1][0] == "Maths"
assert abs(par_matiere[1][1] - 14.5) < 1e-9

# HAVING : ne garder que les matières dont la moyenne dépasse 15
bonnes = cur.execute(
    "SELECT matiere, AVG(note) AS moyenne FROM notes "
    "GROUP BY matiere HAVING moyenne > 15"
).fetchall()
assert bonnes == [("Info", 17.625)]

# HAVING sur un comptage
grosses = cur.execute(
    "SELECT matiere FROM notes GROUP BY matiere HAVING COUNT(*) >= 5"
).fetchall()
assert grosses == [("Maths",)]

# nombre d'élèves par classe
par_classe = dict(cur.execute(
    "SELECT classe, COUNT(*) FROM eleves GROUP BY classe ORDER BY classe"
).fetchall())
assert par_classe == {"MPSI": 2, "PCSI": 2, "PSI": 2}


**Pièges.** Utiliser `WHERE AVG(note) > 15` (interdit : agrégat dans `WHERE`) → il faut `HAVING`.

**Astuce.** Ordre logique : `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY` → `LIMIT`.


## 6.6 Jointures `JOIN` et `LEFT JOIN`

**Définition.** `JOIN` combine deux tables sur une condition ; `LEFT JOIN` conserve toutes les lignes de gauche, même sans correspondance.

**Quand l'utiliser.** `JOIN` pour les correspondances ; `LEFT JOIN` pour repérer les absences (élève sans note).


In [ ]:
jointure = cur.execute("""
    SELECT e.nom, n.matiere, n.note
    FROM eleves AS e
    JOIN notes AS n ON e.id = n.eleve_id
    ORDER BY e.nom, n.matiere
""").fetchall()

assert len(jointure) == 9
assert jointure[0] == ("Amine", "Info", 18.0)
assert jointure[1] == ("Amine", "Maths", 16.5)

# LEFT JOIN : Farah n'a aucune note, elle apparaît quand même
compte = dict(cur.execute("""
    SELECT e.nom, COUNT(n.id)
    FROM eleves AS e
    LEFT JOIN notes AS n ON e.id = n.eleve_id
    GROUP BY e.id
    ORDER BY e.nom
""").fetchall())

assert compte["Amine"] == 2
assert compte["Farah"] == 0
assert len(compte) == 6


**Pièges.** Oublier la condition `ON` (produit cartésien) ; utiliser `JOIN` au lieu de `LEFT JOIN` et perdre les élèves sans note.

**Astuce.** `COUNT(n.id)` (et non `COUNT(*)`) compte les vraies correspondances dans un `LEFT JOIN`.


## 6.7 Requêtes paramétrées (sécurité)

**Définition.** On place des `?` dans la requête et on passe les valeurs dans un tuple : sqlite3 les échappe.

**Quand l'utiliser.** Toujours, dès qu'une valeur vient de l'utilisateur (formulaire, fichier, API).


In [ ]:
resultat = cur.execute(
    "SELECT nom FROM eleves WHERE classe = ? AND age >= ? ORDER BY nom",
    ("PSI", 18),
).fetchall()
assert resultat == [("Farah",)]

# une chaîne d'injection est traitée comme une simple valeur
injection = cur.execute(
    "SELECT nom FROM eleves WHERE nom = ?", ("x' OR '1'='1",)
).fetchall()
assert injection == []


**Pièges.** Ne **jamais** faire `"WHERE nom = '" + nom + "'"` : c'est une faille d'injection SQL.

**Astuce.** Les paramètres `?` fonctionnent aussi pour `INSERT`, `UPDATE` et `DELETE`.


## 6.8 `UPDATE`, `DELETE` et transactions

**Définition.** `UPDATE` modifie des lignes, `DELETE` en supprime ; `commit` valide, `rollback` annule depuis le dernier commit.

**Quand l'utiliser.** Correction de données, purge ; `rollback` pour annuler une opération risquée.


In [ ]:
# UPDATE validé
cur.execute("UPDATE eleves SET age = ? WHERE nom = ?", (20, "Driss"))
conn.commit()
assert cur.execute("SELECT age FROM eleves WHERE nom = 'Driss'").fetchone()[0] == 20

# UPDATE annulé par rollback
cur.execute("UPDATE eleves SET age = 99 WHERE nom = 'Amine'")
conn.rollback()
assert cur.execute("SELECT age FROM eleves WHERE nom = 'Amine'").fetchone()[0] == 18

# DELETE validé
cur.execute("DELETE FROM notes WHERE note < ?", (12,))
conn.commit()
assert cur.execute("SELECT COUNT(*) FROM notes").fetchone()[0] == 8


**Pièges.** Oublier `commit` (modification perdue à la fermeture) ; `DELETE` sans `WHERE` vide la table.

**Astuce.** Faire un `rollback` explicite en cas d'erreur dans un `try/except` pour laisser la base cohérente.


## 6.9 Fermer la connexion

**Définition.** `conn.close()` libère la connexion et détruit la base `:memory:`.

**Quand l'utiliser.** Toujours en fin de script ; `with sqlite3.connect(...) as conn:` gère commit/rollback automatiquement.


In [ ]:
conn.close()
print("connexion fermée — la base en mémoire est détruite")


**Pièges.** Réutiliser un curseur après `close()` lève `ProgrammingError`.

**Astuce.** Le contexte `with sqlite3.connect(...) as conn:` commit en cas de succès et rollback en cas d'exception.


## Récapitulatif

| Clause | Rôle |
|---|---|
| `SELECT … FROM` | choisir colonnes et table |
| `WHERE` | filtrer les lignes |
| `ORDER BY … LIMIT` | trier puis couper |
| `GROUP BY … HAVING` | agréger puis filtrer les groupes |
| `JOIN` / `LEFT JOIN` | combiner deux tables |
| `?` | paramètre sûr (anti-injection) |
| `commit` / `rollback` | valider / annuler |
